# humor-me decisions: the model track (judge, SFT, DPO)

Evidence for the trained joke model: the judge that labels preference pairs, then SFT and DPO. Created with its first experiment (Exp 36). Experiment numbers are global across the `decisions*.ipynb` notebooks; the index is [decisions.ipynb](decisions.ipynb).

**How to use it**
- Results load from `reports/metrics/*.json` (cached by parameters), so nothing recomputes unless you ask.
- Every experiment is a `run(**params)` called through `sim.cache.run_or_load`. Set `RERUN = True` after the import in the setup cell to recompute everything.
- Every claim has a **✅ / ❌ check** under it. If a change flips one to ❌, the decision it backs needs revisiting.
- Before committing: `python scripts/run_notebook.py` (restart + run all, clean metadata).

Rules: logic lives in modules; cells only call, display and check. Vocabulary: [glossary](../docs/wiki/glossary.md).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"))
from decisions_common import RERUN, check, show

## Exp 36 Does the judge agree with you on same-writer pairs?

**Decision (D-121, carrying out D-102):** DPO needs thousands of "which bit is funnier" labels, and only a judge can give that many. Before it labels anything it must agree with you on two bits of one request written by the same model (closer than Exp 24's cross-model ballot). You rank 120 pairs: 60 teach the judge your taste, 60 are held back as the test.
**Where it fits:** the model track, between the live writer's bits and DPO training. The pairs come from the live writer (`python -m sim.dpo_pairs write`), half of them with tonight's context.
**Question:** on the test pairs it answers the same way in both orders, does the judge pick your bit?
**Method:** `python -m sim.dpo_pairs pool`, `write --split ranking`, `rank` (in sittings), `judge`, `score --record`. The judge is Claude Haiku 5.5, thinking off (the user's choice), taught with the 60 teach pairs as examples.
**Bar (D-102, with the user's rule for inconsistent pairs, set before any data):** a test pair whose two orders disagree gets no label. Let c be the pairs answered the same way in both orders and a those agreeing with you. Pass iff c >= 30 and the Wilson 95% lower end of a/c is above 50% (a >= 21 at c = 30, a >= 38 at c = 60). Reported, no bar: a, c, the share of pairs whose orders disagree, agreement per tonight half; beside them the strict count and the half-credit rate.
**What is made up:** nothing from `audience.yaml`. Pool sizes (240 and 800) and the 30-pair coverage floor are judgment calls, not measured. One day of headlines for the tonight half: the training pool is rebuilt with 3+ days before labelling.
**Outcome (2026-11-27): the bar was not met.** The judge answered 46 of 60 test pairs the same way in both orders (bar 30, met) but picked your bit on 24 of those 46: 52%, likely range 38% to 66% (bar: low end above 50%). Haiku 5.5 does not predict your taste on same-writer pairs, so DPO does not run on its labels. Exp 37 (D-122) tries Opus 5.5 on the same pairs.
**How to read:** the checks first (they hold before any answer), then the table. The outcome is shown, not claimed.

In [2]:
from sim import dpo_pairs as dp
from sim import judge_calibration as jc

# Checks that hold before any answer: the bar and the design are the ones decided.
least = lambda c: next(a for a in range(c + 1) if jc.wilson(a, c)[0] > 0.5)
check("The gate is the user's: at least 30 of 60 test pairs answered the same way in both orders, and among those the Wilson lower end above 50% (21 of 30, 38 of 60 is the least)",
      (dp.MIN_CONSISTENT, dp.N_TEST) == (30, 60) and (least(30), least(60)) == (21, 38)
      and dp.passes(21, 30) and not dp.passes(20, 30) and not dp.passes(29, 29) and dp.passes(38, 60) and not dp.passes(37, 60))
check("120 decided pairs are 4 cells of 30: teach and test, tonight half and plain half",
      dp.N_PAIRS == 120 and dp.CELLS == (("teach", "tonight"), ("test", "tonight"), ("teach", "plain"), ("test", "plain")) and dp.PER_CELL == 30)

# The pool on a made-up day (no stored headline text): deterministic, split disjoint, every cell big enough.
day = {"date": "2000-01-01", "approved": [f"gist {i}" for i in range(5)], "facts": [], "market": "up", "names": [], "people": []}
pool = dp.build_pool([day])
check("The pool is deterministic for a seed and its sizes are the decided ones",
      dp.build_pool([day]) == pool and dp.sizes(pool) == {"ranking/tonight": 120, "ranking/plain": 120, "train/tonight": 400, "train/plain": 400})
dp.assert_disjoint(pool)  # raises if a ranking or training prompt repeats Exp 24's ballot or Exp 26, 27, 30, or the splits overlap
check("No ranking or training prompt repeats one of Exp 24's ballot or Exp 26, 27 and 30, and the two splits share none (assert_disjoint passed)", True)
check("Roles are fixed in the pool before any answer: 60 prompts of each role in each tonight half of the ranking split",
      all(sum(p["split"] == "ranking" and p["half"] == h and p["role"] == r for p in pool["prompts"]) == 60 for h in dp.HALVES for r in dp.ROLES))
check("Headline topics sit on bits 2 and 6 only, and only in the tonight half",
      {p["bit"] for p in pool["prompts"] if p["gist"]} == {2, 6} and all(p["half"] == "tonight" for p in pool["prompts"] if p["gist"]))

res = dp.stored()
if res is None:
    show("**Pending.** Rank 120 pairs (`python -m sim.dpo_pairs rank`), then `judge` and `score --record` (Ollama up for `write`, `ANTHROPIC_API_KEY` set for `judge`), then re-run this notebook.")
else:
    show(dp.table(res))
    show(f"**Script verdict: `{res['verdict']}`.** Judge: {res['judge']['model']}, taught with {res['judge']['teach_pairs']} pairs.")
    rule = "pass" if dp.passes(res["agree_consistent"], res["consistent"]) else "fail"
    check("The recorded verdict is what the fixed rule gives for the recorded numbers",
          res["verdict"] == (rule if res["n_test"] == dp.N_TEST and not res["missing_pairs"] else "incomplete"))
    check("The two halves add up to the test pairs", sum(h["pairs"] for h in res["by_half"].values()) == res["n_test"])
    check("Consistent pairs are a subset of the test pairs, and the agreeing ones a subset of those", res["agree_consistent"] <= res["consistent"] <= res["n_test"])
    check("Exp 36 outcome: gate not met, 24 of 46 consistent pairs agree, low end of the likely range 38%",
          (res["agree_consistent"], res["consistent"], res["verdict"]) == (24, 46, "fail") and round(res["wilson95"][0], 2) == 0.38
          and res["consistent"] >= dp.MIN_CONSISTENT and res["agree_consistent"] < least(res["consistent"]))
    check("The ranking left 12 ties and 12 skips among the 144 answers, and the re-rank for self-agreement was not run",
          (res["ties"], res["skips"]) == (12, 12) and res["self_agreement"]["status"] == "not run")

✅ **The gate is the user's: at least 30 of 60 test pairs answered the same way in both orders, and among those the Wilson lower end above 50% (21 of 30, 38 of 60 is the least)**

✅ **120 decided pairs are 4 cells of 30: teach and test, tonight half and plain half**

✅ **The pool is deterministic for a seed and its sizes are the decided ones**

✅ **No ranking or training prompt repeats one of Exp 24's ballot or Exp 26, 27 and 30, and the two splits share none (assert_disjoint passed)**

✅ **Roles are fixed in the pool before any answer: 60 prompts of each role in each tonight half of the ranking split**

✅ **Headline topics sit on bits 2 and 6 only, and only in the tonight half**

... (3 outputs left out)

✅ **The two halves add up to the test pairs**

✅ **Consistent pairs are a subset of the test pairs, and the agreeing ones a subset of those**

✅ **Exp 36 outcome: gate not met, 24 of 46 consistent pairs agree, low end of the likely range 38%**

✅ **The ranking left 12 ties and 12 skips among the 144 answers, and the re-rank for self-agreement was not run**

## Exp 37 Does Claude Opus 5.5 agree with you on the same pairs?

**Decision (D-122, after Exp 36 failed):** try one other judge, Claude Opus 5.5, on the same 60 teach and 60 test pairs rather than rank fresh ones (the ranking took 12 to 16 hours).
**Where it fits:** the model track, in place of the Haiku judge if it passes; DPO needs a judge that agrees with you on same-writer pairs (D-102).
**Question:** on the test pairs it answers the same way in both orders, does Opus 5.5 pick your bit?
**Method:** `python -m sim.dpo_pairs judge --model opus --env-file backend/.env`, then `score --model opus --record`. Same prompt, teach examples, both orders and effort as Exp 36. Opus 5.5 cannot run with thinking off, so its thinking is on. Own judge cache and own record.
**Bar (set before any Opus answer):** the test pairs are used a second time, so a Bonferroni correction for two looks: at least 30 pairs answered the same way in both orders, and among those a likely range (97.5%, z = 2.2414) whose low end is above 50%. Least that clears: 22 of 30, 31 of 46, 39 of 60. A judge truly right 65% of the time in each order passes 69.5% of the time. A judgment call on the correction method, not measured.
**What is made up:** nothing from `audience.yaml`. The correction method, Opus's timeouts and the cost estimate's output tokens are judgment calls.
**Outcome:** pending. You run Exp 37 on this branch; the table below fills in then. A pass lets Opus label the training pool; a fail means DPO does not run. The Exp 36 fail stays recorded either way.
**How to read:** the checks first (they hold before any answer), then the table.

In [3]:
from math import comb
from statistics import NormalDist

z2 = dp.Z["opus"]
least_at = lambda z, c: next(a for a in range(c + 1) if jc.wilson(a, c, z)[0] > 0.5)
check("The second look's bar is Bonferroni for two looks: z is the 97.5% two-sided value, 2.2414",
      round(NormalDist().inv_cdf(1 - 0.05 / 4), 4) == z2 and dp.Z["haiku"] == 1.96)
check("Least agreeing pairs: 22 of 30, 31 of 46, 39 of 60 (Exp 36's bar needed 21, 30, 38)",
      [least_at(z2, c) for c in (30, 46, 60)] == [22, 31, 39] and [least_at(dp.Z["haiku"], c) for c in (30, 46, 60)] == [21, 30, 38]
      and dp.passes(22, 30, z2) and not dp.passes(21, 30, z2) and not dp.passes(29, 29, z2) and dp.passes(39, 60, z2) and not dp.passes(38, 60, z2))
check("Haiku's recorded result fails Exp 37's bar too (24 of 46 needs 31), so the stricter bar never rescues it", not dp.passes(24, 46, z2))


def chance(p, z):
    q, r = p * p + (1 - p) ** 2, p * p / (p * p + (1 - p) ** 2)
    out = 0.0
    for c in range(dp.MIN_CONSISTENT, dp.N_TEST + 1):
        a0 = next((a for a in range(c + 1) if jc.wilson(a, c, z)[0] > 0.5), None)
        if a0 is not None:
            out += comb(60, c) * q ** c * (1 - q) ** (60 - c) * sum(comb(c, a) * r ** a * (1 - r) ** (c - a) for a in range(a0, c + 1))
    return out


check("Chance of passing at 50 / 65 / 70 / 75% right per order: 0.7% / 69.5% / 90.7% / 98.2% (Exp 36's bar: 1.4% / 73.8% / 91.2% / 98.2%)",
      [round(chance(p, z2), 3) for p in (.5, .65, .7, .75)] == [0.007, 0.695, 0.907, 0.982]
      and [round(chance(p, dp.Z["haiku"]), 3) for p in (.5, .65, .7, .75)] == [0.014, 0.738, 0.912, 0.982])
check("Opus has its own model id, its own cache folder and its own record, and Haiku stays the default",
      dp.judge_model("opus")[1] == "claude-opus-5-5" and dp.judge_model()[1] == "claude-haiku-5-5"
      and dp.RECORD == {"haiku": "dpo_judge_gate", "opus": "dpo_judge_gate_opus"})

res37 = dp.stored("opus")
if res37 is None:
    show("**Pending.** Run `python -m sim.dpo_pairs judge --model opus --env-file backend/.env` and `score --model opus --record`, then re-run this notebook.")
else:
    show(dp.table(res37, "opus"))
    show(f"**Script verdict: `{res37['verdict']}`.** Judge: {res37['judge']['model']}, taught with {res37['judge']['teach_pairs']} pairs.")
    rule37 = "pass" if dp.passes(res37["agree_consistent"], res37["consistent"], z2) else "fail"
    check("The recorded verdict is what the corrected rule gives for the recorded numbers",
          res37["verdict"] == (rule37 if res37["n_test"] == dp.N_TEST and not res37["missing_pairs"] else "incomplete"))
    check("The judge is Opus 5.5", res37["judge"]["model"] == "claude-opus-5-5")

✅ **The second look's bar is Bonferroni for two looks: z is the 97.5% two-sided value, 2.2414**

✅ **Least agreeing pairs: 22 of 30, 31 of 46, 39 of 60 (Exp 36's bar needed 21, 30, 38)**

✅ **Haiku's recorded result fails Exp 37's bar too (24 of 46 needs 31), so the stricter bar never rescues it**

✅ **Chance of passing at 50 / 65 / 70 / 75% right per order: 0.7% / 69.5% / 90.7% / 98.2% (Exp 36's bar: 1.4% / 73.8% / 91.2% / 98.2%)**

✅ **Opus has its own model id, its own cache folder and its own record, and Haiku stays the default**

**Pending.** Run `python -m sim.dpo_pairs judge --model opus --env-file backend/.env` and `score --model opus --record`, then re-run this notebook.